# Wave 2 Final — Account 2: LLaVA AI2D Remaining + LLaVA Pairing 13 + LLaVA Open-Ended

**Tasks Executed:**
1. **LLaVA-1.5-7B AI2D Remaining Shards** (patch + swapblend) — ~1.6h
2. **LLaVA-1.5-7B A-OKVQA Pairing Seed 13** (swapblend, N=500) — ~1.4h
3. **LLaVA-1.5-7B Open-Ended** (A-OKVQA, 2 corruptions: `gaussian,swapblend`, N=500) — ~8.5h

**Total Projected Runtime:** ~11.5 hours (safely under 12h Kaggle ceiling).

> ⚠️ Run with **Save & Run All** (Commit mode), GPU T4 x2, Internet ON.
> Note: Intermediate zips are created after each completed job so shards are preserved even in case of early exit.


In [ ]:
import os
import sys
import time
import subprocess

print("=" * 70)
print("WAVE 2 FINAL - ACCOUNT 2: LLAVA AI2D + PAIRING 13 + OPEN-ENDED")
print("=" * 70)

# STEP 1: FIX CUDA CONFLICT & INSTALL DEPENDENCIES
print("\n>>> Step 1: Fixing CUDA conflict and installing dependencies...")
subprocess.run("pip uninstall -y torchaudio", shell=True)
subprocess.run(
    "pip install -q -U transformers accelerate datasets pillow torchvision bitsandbytes",
    shell=True, check=True
)

# STEP 2: CLONE OR UPDATE REPOSITORY (EXTENSION BRANCH)
print("\n>>> Step 2: Synchronizing GitHub repository (extension branch)...")
repo_dir = "/kaggle/working/reliance-utility"
if not os.path.exists(repo_dir):
    subprocess.run(
        f"git clone https://github.com/Stealth768/reliance-utility.git {repo_dir}",
        shell=True, check=True
    )
else:
    print("Repository exists on disk. Resetting and pulling latest...")
    subprocess.run(f"cd {repo_dir} && git reset --hard HEAD", shell=True, check=True)

subprocess.run(
    f"cd {repo_dir} && git fetch --all --tags && git checkout extension && git pull origin extension",
    shell=True, check=True
)

import torch
assert torch.cuda.is_available(), "GPU accelerator not detected!"
print(f"CUDA GPU Verified: {torch.cuda.get_device_name(0)}")

out_ai2d = "/kaggle/working/results/raw/vlm_ai2d"
out_pair = "/kaggle/working/results/raw/vlm_pairings"
out_open = "/kaggle/working/results/raw/vlm_open"
os.makedirs(out_ai2d, exist_ok=True)
os.makedirs(out_pair, exist_ok=True)
os.makedirs(out_open, exist_ok=True)

def run_guarded_job(job_name, check_cmd, prod_cmd):
    print("\n" + "=" * 70)
    print(f"STARTING GUARDED JOB: {job_name}")
    print("=" * 70)
    print(f">> Executing pre-flight check: {check_cmd}", flush=True)
    check_res = subprocess.run(check_cmd, shell=True)
    if check_res.returncode != 0:
        print(f"\n[FATAL ERROR] Pre-flight check failed with code {check_res.returncode}!")
        print(f"Stopping job '{job_name}' immediately. Refusing to run full production.")
        sys.exit(check_res.returncode)
    print(f"[CHECK PASSED] Launching production run: {prod_cmd}", flush=True)
    t0 = time.time()
    subprocess.run(prod_cmd, shell=True, check=True)
    elapsed_min = (time.time() - t0) / 60
    print(f"[JOB COMPLETED] Finished {job_name} in {elapsed_min:.1f} min", flush=True)

t_global_start = time.time()

# ----------------------------------------------------------------------
# JOB 1: LLaVA-1.5-7B AI2D Remaining (patch + swapblend) (~1.6 hours)
# Note: --corruptions patch,swapblend runs only the remaining corruptions
# ----------------------------------------------------------------------
run_guarded_job(
    job_name="LLaVA-1.5-7B AI2D (patch + swapblend)",
    check_cmd=f"cd {repo_dir} && python -u scripts/run_vlm_pilot.py --model llava-hf/llava-1.5-7b-hf --dataset ai2d --assistant_prefix answer_colon --check",
    prod_cmd=(
        f"cd {repo_dir} && python -u scripts/run_vlm_pilot.py "
        f"--model llava-hf/llava-1.5-7b-hf "
        f"--dataset ai2d "
        f"--assistant_prefix answer_colon "
        f"--corruptions patch,swapblend "
        f"--data_ids results/derived/ai2d_main_ids.json "
        f"--n 500 --seed 0 --shard_size 25 "
        f"--out {out_ai2d}"
    )
)

print("\n>>> Checkpoint zip: packaging completed AI2D shards...")
subprocess.run(f"cd /kaggle/working && zip -r -q checkpoint_acc2_ai2d.zip results/raw/vlm_ai2d", shell=True)
print("Checkpoint zip created: checkpoint_acc2_ai2d.zip")

# ----------------------------------------------------------------------
# JOB 2: LLaVA-1.5-7B A-OKVQA Pairing Seed 13 (~1.4 hours)
# ----------------------------------------------------------------------
run_guarded_job(
    job_name="LLaVA-1.5-7B A-OKVQA Pairing Seed 13",
    check_cmd=f"cd {repo_dir} && python -u scripts/run_vlm_pilot.py --model llava-hf/llava-1.5-7b-hf --dataset aokvqa --assistant_prefix answer_colon --check",
    prod_cmd=(
        f"cd {repo_dir} && python -u scripts/run_vlm_pilot.py "
        f"--model llava-hf/llava-1.5-7b-hf "
        f"--dataset aokvqa "
        f"--assistant_prefix answer_colon "
        f"--corruptions swapblend "
        f"--pairing_seed 13 "
        f"--seed 0 --shard_size 25 "
        f"--out {out_pair}"
    )
)

print("\n>>> Checkpoint zip: packaging completed pairing shards...")
subprocess.run(f"cd /kaggle/working && zip -r -q checkpoint_acc2_pair13.zip results/raw/vlm_pairings", shell=True)
print("Checkpoint zip created: checkpoint_acc2_pair13.zip")

# ----------------------------------------------------------------------
# JOB 3: LLaVA-1.5-7B Open-Ended (gaussian, swapblend) (~8.5 hours)
# ----------------------------------------------------------------------
run_guarded_job(
    job_name="LLaVA-1.5-7B Open-Ended (gaussian + swapblend)",
    check_cmd=f"cd {repo_dir} && python -u scripts/run_vlm_open.py --model llava-hf/llava-1.5-7b-hf --assistant_prefix answer_colon --check",
    prod_cmd=(
        f"cd {repo_dir} && python -u scripts/run_vlm_open.py "
        f"--model llava-hf/llava-1.5-7b-hf "
        f"--assistant_prefix answer_colon "
        f"--corruptions gaussian,swapblend "
        f"--data_ids results/derived/main_ids.json "
        f"--n 500 --seed 0 --shard_size 25 "
        f"--out {out_open}"
    )
)

# ----------------------------------------------------------------------
# FINAL PACKAGING
# ----------------------------------------------------------------------
print("\n>>> Step 5: Packaging all completed shards into final zip...")
final_zip = "account2_wave2_final.zip"
subprocess.run(
    f"cd /kaggle/working && zip -r -q {final_zip} results/raw/vlm_ai2d results/raw/vlm_pairings results/raw/vlm_open",
    shell=True, check=True
)

zip_path = f"/kaggle/working/{final_zip}"
size_mb = os.path.getsize(zip_path) / (1024 * 1024)
total_hours = (time.time() - t_global_start) / 3600

print("=" * 70)
print(f"SUCCESS! Account 2 completed in {total_hours:.2f} hours.")
print(f"Final archive: {zip_path} ({size_mb:.2f} MB)")
print("=" * 70)
print("NOTE: Download this archive directly from the 'Output' tab of your committed Kaggle notebook version.")
